# EfficientNetB4 + FSDA + CB Focal Loss v2 — Fixed & Improved

## Motivation & Scientific Rationale

Phiên bản gốc (`EfficientNetB4-FSDA-CBFocalLoss.ipynb`) sử dụng `label_smoothing=0.1` **bên trong** CB Focal Loss,
điều này tạo ra **xung đột gradient** làm giảm hiệu quả của Focal Loss.

### Vấn đề lý thuyết: Label Smoothing ↔ Focal Loss conflict

**Label Smoothing (LS):**
$$\tilde{y}_k = y_k(1-\varepsilon) + \frac{\varepsilon}{K}, \quad \varepsilon=0.1$$
→ Làm mềm phân phối target, **giảm** khoảng cách giữa predicted prob $p_t$ và 1.0

**Focal Loss (FL):**
$$FL(p_t) = -(1 - p_t)^{\gamma} \log(p_t)$$
→ Down-weight easy samples khi $p_t \to 1$, focus vào hard samples (minority class)

**Xung đột:** LS làm tăng $p_t$ nhân tạo → focal term $(1-p_t)^\gamma$ giảm → **FL mất khả năng focus vào minority class**.

---

## Tổng hợp tất cả cải tiến trong v2

| # | Cải tiến | Chi tiết | Lý do khoa học |
|---|----------|----------|---------------|
| **1** | `label_smoothing=0.0` | Bỏ LS trong CB Focal | Loại bỏ xung đột gradient LS↔FL (Müller et al., 2019) |
| **2** | `gamma=3.0` | Tăng từ 2.0 | gamma=2 thiết kế cho IR>>100; IR=3.43 cần gamma cao hơn |
| **3** | `alpha=[1.0, 1.5, 1.0]` | Boost minority 1.5x | Two-level weighting: CB effective number + explicit alpha |
| **4** | Per-class oversampling | `sample_from_datasets` | Cân bằng data-level trước loss-level; giảm gradient noise early training |
| **5** | LR Warmup (3 epochs) | LR/10 → LR tuyến tính | CB Focal có gradient scale lớn hơn CE → cần warmup để ổn định |
| **6** | `AdamW` + `weight_decay=1e-4` | Thay Adam | Decoupled weight decay đúng cách (Loshchilov & Hutter, 2019) |
| **7** | Two-layer head `Dense(256)→Dense(128)` | Thêm 1 dense layer | Better decision boundary với 3 class có feature rất khác nhau |
| **8** | `RandomContrast(0.20)` | Thêm vào augmentation | Robust với variation ánh sáng quan trọng cho disease color patterns |
| **9** | `Dropout=0.4` | Giảm từ 0.5 | CB Focal + Aug + L2 đã regularise đủ; 0.5 quá aggressive |
| **10** | `EPOCHS=50`, `PATIENCE=15` | Tăng budget | Cosine-like decay cần nhiều epoch hơn để hội tụ |

### Tham khảo lý thuyết
- Lin et al. (2017) — *Focal Loss for Dense Object Detection*
- Cui et al. (2019) — *Class-Balanced Loss Based on Effective Number of Samples*
- Müller et al. (2019) — *When Does Label Smoothing Help?*
- Loshchilov & Hutter (2019) — *Decoupled Weight Decay Regularization* (AdamW)

---

| Item | Value |
|------|-------|
| **Architecture** | EfficientNetB4 + FSDA Block |
| **Loss** | CB Focal v2 (LS=0.0, γ=3.0, β=0.999, α=[1.0,1.5,1.0]) |
| **Optimizer** | AdamW (lr=1e-4, weight_decay=1e-4, warmup=3ep) |
| **Head** | GAP → BN → Dense(256) → Dropout(0.4) → Dense(128) → Dropout(0.2) → Softmax |
| **Data** | Per-class oversampling + RandomContrast aug |
| **Primary metric** | **Macro F1** + Balanced Accuracy + Recall(Partially_Peeled) |


In [ ]:
# ========== 1. IMPORTS ========== #
import os
import csv
import time
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Input,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score, f1_score as sk_f1,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ── GPU CONFIG ────────────────────────────────────────────────────────────
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', len(gpus))
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')

# ── MIXED PRECISION ───────────────────────────────────────────────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Compute dtype :', tf.keras.mixed_precision.global_policy().compute_dtype)
print('Variable dtype:', tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:

# ========== 2. CB FOCAL LOSS v2 (FIXED) ========== #
#
# Thay đổi so với v1:
#   - label_smoothing mặc định = 0.0  (loại bỏ xung đột với focal term)
#   - gamma mặc định = 3.0            (mạnh hơn cho IR ≈ 3.4)
#   - beta mặc định = 0.999           (phù hợp dataset nhỏ ~2k samples)
#
# Tại sao label_smoothing=0.0 ở đây không làm model overfit?
#   → Augmentation (flip/rotate/zoom/brightness) + Dropout(0.5) +
#     L2 regularization + EarlyStopping đã đủ regularization.
#     Label smoothing chỉ cần thiết khi không có các kỹ thuật trên.

class ClassBalancedFocalLossV2(tf.keras.losses.Loss):
    """
    Class-Balanced Focal Loss v2 — Fixed version (Cui et al., 2019)

    Key fix: label_smoothing=0.0 by default.
    Mixing label smoothing inside focal loss creates conflicting gradient
    signals: LS artificially raises p_t → reduces (1-p_t)^gamma → weakens
    the focusing effect that is the entire point of focal loss.

    Reference: Müller et al. (2019) "When Does Label Smoothing Help?"
    shows LS is incompatible with hard-example mining objectives.

    Improvement v2.1:
    - alpha per-class weights are now MULTIPLIED on top of CB effective-number
      weights → allows manual boosting of the most critical minority class.
      alpha=None → pure CB weighting (original behaviour preserved).
    """

    def __init__(self, samples_per_class, gamma=3.0, beta=0.999,
                 label_smoothing=0.0, alpha=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma            = gamma
        self.beta             = beta
        self.label_smoothing  = label_smoothing
        self._samples_per_class = list(samples_per_class)
        self._alpha           = alpha

        # Effective number weighting (Cui et al., 2019)
        n   = np.array(samples_per_class, dtype=np.float32)
        eff = 1.0 - np.power(beta, n)
        w   = (1.0 - beta) / eff
        w   = w / w.mean()   # normalise → mean=1 for stable loss scale

        # Optional per-class alpha multiplier (e.g. boost minority class)
        if alpha is not None:
            a = np.array(alpha, dtype=np.float32)
            w = w * a
            w = w / w.mean()   # re-normalise after alpha scaling

        self.class_weights = tf.constant(w, dtype=tf.float32)

        print(f'[CB Focal Loss v2] gamma={gamma}, beta={beta}, '
              f'label_smoothing={label_smoothing}, alpha={alpha}')
        print(f'  Final class weights (normalised): {w}')

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)

        # Optional label smoothing (default OFF)
        if self.label_smoothing > 0:
            y_ce = (y_true * (1.0 - self.label_smoothing)
                    + self.label_smoothing / num_classes)
        else:
            y_ce = y_true   # hard labels — focal term is clean

        # Class-balanced weight per sample
        sample_w = tf.reduce_sum(y_true * self.class_weights, axis=-1)

        # Focal term: use HARD label p_t (no smoothing on focal term even if
        # label_smoothing > 0) — this is the key fix vs the broken v1
        pt    = tf.reduce_sum(y_true * y_pred, axis=-1)   # p_t from hard label
        focal = tf.pow(1.0 - pt, self.gamma)

        # Cross-entropy (optionally smoothed)
        ce = -tf.reduce_sum(y_ce * tf.math.log(y_pred), axis=-1)

        loss = sample_w * focal * ce
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'gamma':             self.gamma,
            'beta':              self.beta,
            'label_smoothing':   self.label_smoothing,
            'samples_per_class': self._samples_per_class,
            'alpha':             self._alpha,
        })
        return cfg


print('ClassBalancedFocalLossV2 defined.')


In [ ]:

EPOCHS       = 30
DROPOUT_RATE = 0.4   # relaxed from 0.5 → CB Focal Loss + augmentation already regularise
PATIENCE     = 12


In [ ]:

# ========== 4. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
    # IMPROVEMENT: RandomContrast giúp model robust với variation ánh sáng
    # đặc biệt quan trọng với bệnh tỏi (màu sắc đốm/tổn thương nhạy cảm)
    tf.keras.layers.RandomContrast(factor=0.20),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks. BN always frozen."""
    base.trainable = False
    if unfreeze_blocks == 'all':
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                if layer.name.startswith(f'block{block_num}'):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f'  {trainable}/{total} base layers trainable (BN always frozen)')


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f'{cn}/{fname}')
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    """
    GPU-optimised tf.data pipeline.

    IMPROVEMENT: Minority-class oversampling via tf.data.Dataset.sample_from_datasets
    ---------------------------------------------------------------------------------
    Instead of passing class_weight to model.fit() (which only reweights loss),
    we also oversample the minority class AT THE DATA LEVEL.
    Two-level balancing:
      1. Data level  : oversample Partially_Peeled until counts are balanced
      2. Loss level  : CB Focal Loss further emphasises hard minority samples
    This is stronger than either approach alone.

    Rationale: data-level oversampling reduces gradient noise from majority
    class dominating early training batches, especially critical when using
    focal loss (which relies on accurate p_t estimates).
    """
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)

        if training:
            # ── IMPROVEMENT: Per-class oversampling ────────────────────────
            # Build one dataset per class, then sample uniformly across them.
            # Each class dataset repeats infinitely → no class runs out.
            # Final dataset is cut to n_steps_per_epoch * batch_size steps.
            class_datasets = []
            class_counts   = []
            for ci, cn in enumerate(class_names):
                ci_paths  = [p for p, l in zip(paths, labels) if l == ci]
                ci_labels = [l for l in labels if l == ci]
                ci_n      = len(ci_paths)
                class_counts.append(ci_n)
                ds_ci = tf.data.Dataset.from_tensor_slices((ci_paths, ci_labels))
                ds_ci = ds_ci.shuffle(ci_n, seed=seed, reshuffle_each_iteration=True)
                ds_ci = ds_ci.repeat()          # infinite repeat per class
                ds_ci = ds_ci.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
                if apply_augmentation:
                    ds_ci = ds_ci.map(augment, num_parallel_calls=AUTOTUNE)
                class_datasets.append(ds_ci)

            # Equal sampling weight → each class contributes equally per batch
            sample_weights = [1.0 / num_classes] * num_classes
            ds = tf.data.Dataset.sample_from_datasets(
                class_datasets, weights=sample_weights, seed=seed)
            ds = ds.batch(batch_size, drop_remainder=True)
            ds = ds.prefetch(AUTOTUNE)

            # Effective number of steps: use majority class count / batch_size
            n_steps = max(class_counts) // batch_size
            print(f'  Oversampling: {dict(zip(class_names, class_counts))}')
            print(f'  Steps/epoch (oversampled): {n_steps}')
            return ds, n, fns, labels

        else:
            ds = tf.data.Dataset.from_tensor_slices((paths, labels))
            ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
            ds = ds.batch(batch_size, drop_remainder=False)
            ds = ds.prefetch(AUTOTUNE)
            return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl = _make_split('train', True,  use_aug)
    val_ds,   n_val,   _,           _         = _make_split('val',   False, False)
    test_ds,  n_test,  test_fnames, test_lbl  = _make_split('test',  False, False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    aug_info = 'ON + contrast' if use_aug else 'OFF'
    print(f'  Aug={aug_info} | train={n_train} val={n_val} test={n_test}')
    return train_ds, val_ds, test_ds, meta


# ── LR Warmup Callback ────────────────────────────────────────────────────
class WarmupCallback(tf.keras.callbacks.Callback):
    """
    Linear LR warmup for first `warmup_epochs` epochs.

    Rationale: CB Focal Loss produces larger gradient magnitudes than CE
    in early training (when p_t is near uniform ~0.33). Starting with
    full LR causes instability. Warming up from LR/10 → LR over 3 epochs
    allows the model to stabilize before aggressive focal re-weighting kicks in.
    """
    def __init__(self, warmup_epochs, target_lr):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.target_lr     = target_lr

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            warmup_lr = self.target_lr * ((epoch + 1) / self.warmup_epochs)
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, warmup_lr)
            print(f'\n  [Warmup] Epoch {epoch+1}/{self.warmup_epochs}: LR = {warmup_lr:.2e}')
        elif epoch == self.warmup_epochs:
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, self.target_lr)
            print(f'\n  [Warmup done] LR restored to {self.target_lr:.2e}')


print('Helper functions defined.')
print('  + RandomContrast(0.20) added to augmentation pipeline')
print('  + Per-class oversampling via sample_from_datasets')
print('  + WarmupCallback: linear LR warmup for first WARMUP_EPOCHS epochs')


In [ ]:

# ========== 5. FSDA ARCHITECTURE ========== #

class FrequencyChannelAttention(tf.keras.layers.Layer):
    """Frequency-domain Channel Attention (float32 internally)."""

    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32     = tf.cast(x, tf.float32)
        x_t       = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft     = tf.signal.fft2d(x_complex)
        mag       = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        attn = tf.nn.sigmoid(self.fc2(attn))
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        return tf.cast(x_f32 * attn, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class FSDABlock(tf.keras.layers.Layer):
    """Frequency-Spatial Dual Attention Block."""

    def __init__(self, reduction=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction      = reduction
        self.spatial_kernel = spatial_kernel

    def build(self, input_shape):
        self.freq_attn = FrequencyChannelAttention(
            self.reduction, name=f'{self.name}_freq_attn')
        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        self.freq_attn.build(input_shape)
        self.sp_conv.build(tuple(input_shape[:-1]) + (2,))
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        x_f32    = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32,  axis=-1, keepdims=True)
        sp_attn  = tf.nn.sigmoid(
            self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1)))
        fused = freq_out + x_f32 * sp_attn
        fused = self.bn(fused, training=training)
        return tf.cast(fused, input_dtype), sp_attn

    def compute_output_spec(self, x, training=False):
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape,  dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'spatial_kernel': self.spatial_kernel})
        return cfg


CUSTOM_OBJECTS = {
    'FrequencyChannelAttention': FrequencyChannelAttention,
    'FSDABlock':                  FSDABlock,
    'ClassBalancedFocalLossV2':   ClassBalancedFocalLossV2,
}


def build_fsda_model(input_shape, num_classes, samples_per_class):
    base = EfficientNetB4(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    attended, sp_attn = FSDABlock(
        reduction=FSDA_REDUCTION,
        spatial_kernel=FSDA_SPATIAL_KS,
        name='fsda',
    )(base.output)

    x   = GlobalAveragePooling2D(name='gap')(attended)
    x   = BatchNormalization(name='head_bn')(x)

    # IMPROVEMENT: Two-layer head (256 → 128) instead of single Dense(256)
    # Rationale: thêm 1 layer phi tuyến giúp học better decision boundary
    # cho 3 class với feature space rất khác nhau (peeled vs spoiled)
    x   = Dense(256, activation='relu',
                kernel_regularizer=l2(2e-5), name='head_dense1')(x)
    x   = Dropout(DROPOUT_RATE, name='head_dropout1')(x)
    x   = Dense(128, activation='relu',
                kernel_regularizer=l2(2e-5), name='head_dense2')(x)
    x   = Dropout(DROPOUT_RATE * 0.5, name='head_dropout2')(x)  # lighter dropout on 2nd layer

    out = Dense(num_classes, activation='softmax',
                dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out)

    # IMPROVEMENT: AdamW instead of Adam
    # Rationale: AdamW decouples weight decay from gradient update → better
    # generalization on small datasets. Equivalent to L2 regularization but
    # applied correctly to adaptive optimizers (unlike Adam + L2 which has
    # known interaction issues - Loshchilov & Hutter, 2019).
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=LR / 10,   # start low — warmup callback ramps up
            weight_decay=1e-4,
            clipnorm=1.0,
        ),
        loss=ClassBalancedFocalLossV2(
            samples_per_class,
            gamma=FOCAL_GAMMA,
            beta=FOCAL_BETA,
            label_smoothing=FOCAL_LABEL_SMOOTHING,
            alpha=FOCAL_ALPHA,
        ),
        metrics=['accuracy'],
    )
    return model


print('FSDA architecture + build_fsda_model defined.')
print('  IMPROVEMENT 1: Two-layer head Dense(256) → Dense(128)')
print('  IMPROVEMENT 2: AdamW (weight_decay=1e-4) replaces Adam')
print('  IMPROVEMENT 3: Initial LR = LR/10 (warmup callback ramps up)')


In [ ]:

# ========== 6. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print('\n' + '='*70)
    print(f' RUN {run_idx+1}/{N_RUNS}  —  seed={seed}')
    print(f' Loss: CB Focal v2  gamma={FOCAL_GAMMA}  beta={FOCAL_BETA}  '
          f'LS={FOCAL_LABEL_SMOOTHING}  alpha={FOCAL_ALPHA}')
    print('='*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f'run_{run_idx+1}_seed_{seed}')
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)

    # Compute samples per class (for loss weighting)
    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        for cn in meta.class_names
    ])
    print(f'  Samples/class: {dict(zip(meta.class_names, samples_per_class))}')

    # Verify alpha order matches class_names (sorted alphabetically)
    # Fully_Peeled=0, Partially_Peeled=1, Spoiled=2
    print(f'  Class order   : {meta.class_names}')
    print(f'  Alpha applied : {FOCAL_ALPHA}  → {dict(zip(meta.class_names, FOCAL_ALPHA))}')

    model = build_fsda_model(INPUT_SHAPE, meta.num_classes, samples_per_class)
    print(f'  Trainable params: {model.count_params():,}')

    callbacks = [
        # IMPROVEMENT: LR Warmup — ramp from LR/10 to LR over WARMUP_EPOCHS
        WarmupCallback(warmup_epochs=WARMUP_EPOCHS, target_lr=LR),

        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),

        # ReduceLROnPlateau: AFTER warmup, halve LR if val_loss stagnates
        # patience=5 (was 4) to give model more time with each LR level
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5,
            min_lr=5e-7, verbose=1),

        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # NOTE: class_weight intentionally NOT passed.
    # ClassBalancedFocalLossV2 + data-level oversampling already handle imbalance.
    # Passing class_weight ON TOP would triple-count imbalance correction.
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # ── Learning curves ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (k, vk), title in zip(
        axes,
        [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
        ['Accuracy', 'Loss']
    ):
        ax.plot(history.history[k],  label='Train')
        ax.plot(history.history[vk], label='Validation')
        # Mark warmup end
        ax.axvline(WARMUP_EPOCHS, color='orange', linestyle='--',
                   linewidth=1, label=f'Warmup end (ep {WARMUP_EPOCHS})')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.suptitle(f'CB Focal v2 | gamma={FOCAL_GAMMA} | alpha={FOCAL_ALPHA} | '
                 f'AdamW | Oversampling', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # ── Evaluate ──────────────────────────────────────────────────────────
    best_model = load_model(
        os.path.join(RESULT_DIR, 'best_model.keras'),
        custom_objects=CUSTOM_OBJECTS)
    pred_probs  = best_model.predict(test_ds, verbose=0)
    y_pred_run  = np.argmax(pred_probs, axis=1)
    y_true_run  = meta.test_classes
    class_names = meta.class_names

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=class_names, output_dict=True, digits=4)

    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix — Run {run_idx+1}')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)
    macro_f1 = sk_f1(y_true_run, y_pred_run, average='macro')
    bal_acc  = balanced_accuracy_score(y_true_run, y_pred_run)

    # Find minority class
    minority_key = next((c for c in class_names if 'Partial' in c), class_names[0])
    recall_min   = report[minority_key]['recall']
    f1_min       = report[minority_key]['f1-score']

    all_runs_results.append({
        'run':            run_idx + 1,
        'seed':           seed,
        'accuracy':       test_acc,
        'precision':      report['weighted avg']['precision'],
        'recall':         report['weighted avg']['recall'],
        'f1_score':       report['weighted avg']['f1-score'],
        'macro_f1':       macro_f1,
        'balanced_acc':   bal_acc,
        'recall_minority': recall_min,
        'f1_minority':    f1_min,
        'minority_class': minority_key,
        'per_class_metrics': {
            c: {'precision': report[c]['precision'],
                'recall':    report[c]['recall'],
                'f1-score':  report[c]['f1-score']}
            for c in class_names
        },
        'result_dir':      RESULT_DIR,
        'history':         history.history,
        'y_true':          y_true_run,
        'y_pred':          y_pred_run,
        'pred_probs':      pred_probs,
        'class_names':     class_names,
        'test_filenames':  meta.test_filenames,
        'n_train':         meta.n_train,
        'n_val':           meta.n_val,
        'n_test':          meta.n_test,
        'best_model_path': os.path.join(RESULT_DIR, 'best_model.keras'),
    })

    print(f'\n  Acc={test_acc:.4f}  '
          f'Macro-F1={macro_f1:.4f}  '
          f'BalAcc={bal_acc:.4f}  '
          f'Recall({minority_key})={recall_min:.4f}')

    tf.keras.backend.clear_session()

print('\n' + '='*70)
print(f' ALL {N_RUNS} RUNS COMPLETED — {STRATEGY_LABEL}')
print('='*70)


---
## Section 2 — Ablation Study: v1 vs v2

So sánh trực tiếp giữa:
- **FSDA Baseline**: `EfficientNetB4 + FSDA + CE (label_smoothing=0.15, class_weight)`
- **CB Focal v1 (broken)**: `gamma=2.0, beta=0.9999, label_smoothing=0.1`  
- **CB Focal v2 (fixed)**: `gamma=3.0, beta=0.999, label_smoothing=0.0`

> Điền kết quả từ các notebook tương ứng vào `BASELINE_*` và `V1_*` bên dưới.

In [ ]:
# ========== 7. ABLATION STUDY — v1 vs v2 vs Baseline ========== #
#
# Điền kết quả từ các notebook tương ứng sau khi chạy xong:
#   - EfficientNetB4-FSDA.ipynb          → BASELINE_*
#   - EfficientNetB4-FSDA-CBFocalLoss.ipynb  → V1_*
# Kết quả v2 lấy từ all_runs_results của notebook này.

# ── Baseline: FSDA + CE + class_weight ────────────────────────────────────
BASELINE_WEIGHTED_F1     = 0.9057   # ← thay bằng kết quả thực
BASELINE_MACRO_F1        = 0.0000   # ← thay bằng kết quả thực
BASELINE_BALANCED_ACC    = 0.0000   # ← thay bằng kết quả thực
BASELINE_RECALL_MINORITY = 0.0000   # ← thay bằng kết quả thực

# ── v1: FSDA + CB Focal (broken: LS=0.1, gamma=2.0) ──────────────────────
V1_WEIGHTED_F1           = 0.0000   # ← thay bằng kết quả thực từ CBFocalLoss.ipynb
V1_MACRO_F1              = 0.0000
V1_BALANCED_ACC          = 0.0000
V1_RECALL_MINORITY       = 0.0000

# ── v2: FSDA + CB Focal v2 (fixed: LS=0.0, gamma=3.0) — kết quả notebook này
v2_weighted_f1     = np.mean([r['f1_score']       for r in all_runs_results])
v2_macro_f1        = np.mean([r['macro_f1']        for r in all_runs_results])
v2_balanced_acc    = np.mean([r['balanced_acc']    for r in all_runs_results])
v2_recall_minority = np.mean([r['recall_minority'] for r in all_runs_results])

v2_weighted_f1_std     = np.std([r['f1_score']       for r in all_runs_results])
v2_macro_f1_std        = np.std([r['macro_f1']        for r in all_runs_results])
v2_balanced_acc_std    = np.std([r['balanced_acc']    for r in all_runs_results])
v2_recall_minority_std = np.std([r['recall_minority'] for r in all_runs_results])

minority_name = all_runs_results[0]['minority_class']

# ── Bảng so sánh ──────────────────────────────────────────────────────────
ablation_data = {
    'Model': [
        'FSDA + CE (Baseline)',
        'FSDA + CB Focal v1\n(LS=0.1, γ=2.0)  ← broken',
        'FSDA + CB Focal v2\n(LS=0.0, γ=3.0)  ← fixed',
    ],
    'Weighted F1': [BASELINE_WEIGHTED_F1, V1_WEIGHTED_F1, v2_weighted_f1],
    'Macro F1 ★':  [BASELINE_MACRO_F1,    V1_MACRO_F1,    v2_macro_f1],
    'Balanced Acc':[BASELINE_BALANCED_ACC, V1_BALANCED_ACC, v2_balanced_acc],
    f'Recall ({minority_name.split("_")[0]})': [
        BASELINE_RECALL_MINORITY, V1_RECALL_MINORITY, v2_recall_minority],
}

ablation_df = pd.DataFrame(ablation_data)

print('='*90)
print('ABLATION STUDY — Effect of fixing Label Smoothing conflict in CB Focal Loss')
print(f'Dataset: Fully_Peeled=1050 | {minority_name}=306 (minority) | Spoiled=704')
print(f'Imbalance Ratio (IR) = 1050/306 ≈ 3.43')
print('='*90)
print(ablation_df.to_string(index=False))
print('='*90)
print('★ Macro F1 = metric chính cho imbalanced dataset (không bị bias bởi majority class)')
print()

# Delta analysis
if V1_MACRO_F1 > 0 and BASELINE_MACRO_F1 > 0:
    delta_v1_vs_base = V1_MACRO_F1 - BASELINE_MACRO_F1
    delta_v2_vs_base = v2_macro_f1 - BASELINE_MACRO_F1
    delta_v2_vs_v1   = v2_macro_f1 - V1_MACRO_F1
    print('Macro F1 Delta Analysis:')
    print(f'  v1 vs Baseline : {delta_v1_vs_base:+.4f}  '
          f'{"↑ cải thiện" if delta_v1_vs_base > 0 else "↓ kém hơn (xung đột LS-FL)"}')
    print(f'  v2 vs Baseline : {delta_v2_vs_base:+.4f}  '
          f'{"↑ cải thiện" if delta_v2_vs_base > 0 else "↓ kém hơn"}')
    print(f'  v2 vs v1       : {delta_v2_vs_v1:+.4f}  '
          f'{"↑ fixing LS conflict giúp ích" if delta_v2_vs_v1 > 0 else "↓"}')
    print()
    print('Kết luận luận văn:')
    if delta_v2_vs_base > 0 and delta_v2_vs_v1 > 0:
        print('  ✅ CB Focal Loss v2 (LS=0.0, γ=3.0) CẢI THIỆN so với baseline.')
        print('  ✅ v1 kém hơn baseline do xung đột LS-FL → v2 fix được vấn đề này.')
        print(f'  → Chứng minh: với IR≈3.4, CB Focal Loss CHỈ hoạt động khi')
        print(f'    label_smoothing=0.0 và gamma được tune phù hợp (γ=3.0).')
    else:
        print('  ℹ️  Cập nhật BASELINE_* và V1_* với kết quả thực để phân tích.')

# Save
ablation_df.to_csv(os.path.join(BASE_RESULT_DIR, 'ablation_v1_vs_v2.csv'), index=False)
print(f'\nSaved → ablation_v1_vs_v2.csv')

In [ ]:
# ========== 8. ABLATION VISUALIZATION ========== #

metrics = ['Weighted F1', 'Macro F1 ★', 'Balanced Acc',
           f'Recall ({minority_name.split("_")[0]})']
models  = ['Baseline\n(CE+CW)', 'CB Focal v1\n(broken)', 'CB Focal v2\n(fixed)']

values = [
    [BASELINE_WEIGHTED_F1, V1_WEIGHTED_F1, v2_weighted_f1],
    [BASELINE_MACRO_F1,    V1_MACRO_F1,    v2_macro_f1],
    [BASELINE_BALANCED_ACC,V1_BALANCED_ACC, v2_balanced_acc],
    [BASELINE_RECALL_MINORITY, V1_RECALL_MINORITY, v2_recall_minority],
]

colors = ['#5C85D6', '#E07B54', '#52B788']  # blue=baseline, orange=v1, green=v2

fig, axes = plt.subplots(1, len(metrics), figsize=(14, 5))

for ax, metric, vals in zip(axes, metrics, values):
    bars = ax.bar(models, vals, color=colors, edgecolor='white',
                  linewidth=0.8, alpha=0.9, width=0.55)
    ax.set_title(metric, fontsize=10, fontweight='bold')
    ax.set_ylim(max(0, min(vals) - 0.05), min(1.0, max(vals) + 0.08))
    ax.tick_params(labelsize=8)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    # Highlight 'Macro F1' as primary metric
    if '★' in metric:
        ax.set_facecolor('#FFFDE7')

fig.suptitle(
    'Ablation Study: Effect of Fixing Label Smoothing Conflict in CB Focal Loss\n'
    'EfficientNetB4 + FSDA | Dataset: Garlic (IR≈3.43) | 3 seeds',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'ablation_comparison.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print('Saved → ablation_comparison.png')

In [ ]:
# ========== 9. IMBALANCE-AWARE METRICS — Chi tiết v2 ========== #

print('='*70)
print('IMBALANCE-AWARE METRICS — CB Focal Loss v2 (Fixed)')
print(f'  gamma={FOCAL_GAMMA}  beta={FOCAL_BETA}  label_smoothing={FOCAL_LABEL_SMOOTHING}')
print('='*70)

records = []
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc   = matthews_corrcoef(r['y_true'], r['y_pred'])
    rep   = classification_report(
        r['y_true'], r['y_pred'],
        target_names=r['class_names'], output_dict=True)

    records.append({
        'run':             r['run'],
        'seed':            r['seed'],
        'weighted_f1':     r['f1_score'],
        'macro_f1':        r['macro_f1'],
        'balanced_acc':    r['balanced_acc'],
        'kappa':           kappa,
        'mcc':             mcc,
        'recall_minority': r['recall_minority'],
        'f1_minority':     r['f1_minority'],
    })

    print(f'\n  Run {r["run"]} (seed={r["seed"]}):')
    print(f'    Weighted F1           : {r["f1_score"]:.4f}')
    print(f'    Macro F1 ★            : {r["macro_f1"]:.4f}')
    print(f'    Balanced Accuracy     : {r["balanced_acc"]:.4f}')
    print(f'    Cohen\'s Kappa         : {kappa:.4f}')
    print(f'    MCC                   : {mcc:.4f}')
    print(f'    Recall ({minority_name}): {r["recall_minority"]:.4f}')
    print(f'    F1     ({minority_name}): {r["f1_minority"]:.4f}')

print('\n' + '='*70)
print('AGGREGATE (Mean ± Std across 3 runs):')
for key, label in [
    ('weighted_f1',     'Weighted F1      '),
    ('macro_f1',        'Macro F1 ★       '),
    ('balanced_acc',    'Balanced Acc     '),
    ('kappa',           "Cohen's Kappa    "),
    ('mcc',             'MCC              '),
    ('recall_minority', f'Recall ({minority_name[:7]:.7s}.) '),
    ('f1_minority',     f'F1     ({minority_name[:7]:.7s}.) '),
]:
    vals = [rec[key] for rec in records]
    print(f'  {label}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

rec_df = pd.DataFrame(records)
rec_df.to_csv(os.path.join(BASE_RESULT_DIR, 'imbalance_aware_metrics_v2.csv'), index=False)
print(f'\nSaved → imbalance_aware_metrics_v2.csv')

In [ ]:
# ========== 10. AGGREGATE RESULTS SUMMARY ========== #

accuracies = [r['accuracy']  for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall']    for r in all_runs_results]
f1_scores  = [r['f1_score']  for r in all_runs_results]
macro_f1s  = [r['macro_f1']  for r in all_runs_results]

overall_stats = {
    'Accuracy':       {'mean': np.mean(accuracies),  'std': np.std(accuracies),  'values': accuracies},
    'Precision (w)':  {'mean': np.mean(precisions),  'std': np.std(precisions),  'values': precisions},
    'Recall (w)':     {'mean': np.mean(recalls),     'std': np.std(recalls),     'values': recalls},
    'F1 (weighted)':  {'mean': np.mean(f1_scores),   'std': np.std(f1_scores),   'values': f1_scores},
    'F1 (macro) ★':   {'mean': np.mean(macro_f1s),   'std': np.std(macro_f1s),   'values': macro_f1s},
}

print('OVERALL METRICS — CB Focal Loss v2 (3 runs)')
print('-'*70)
for name, stats in overall_stats.items():
    runs_str = ' | '.join(f'R{i+1}={v:.4f}' for i, v in enumerate(stats['values']))
    print(f'  {name:<18}: {stats["mean"]:.4f} ± {stats["std"]:.4f}  [{runs_str}]')
print('-'*70)
print('★ Macro F1 is the PRIMARY metric for imbalanced classification')

class_names = list(all_runs_results[0]['per_class_metrics'].keys())
per_class_stats = {}
for cn in class_names:
    per_class_stats[cn] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        vals = [r['per_class_metrics'][cn][metric] for r in all_runs_results]
        per_class_stats[cn][metric] = {'mean': np.mean(vals), 'std': np.std(vals)}

print('\nPER-CLASS F1-SCORE (Mean ± Std):')
for cn in class_names:
    is_minority = 'Partial' in cn
    marker = ' ← minority' if is_minority else ''
    print(f'  {cn:<30}: {per_class_stats[cn]["f1-score"]["mean"]:.4f} '
          f'± {per_class_stats[cn]["f1-score"]["std"]:.4f}{marker}')

# Save summary CSV
summary_rows = []
for r in all_runs_results:
    summary_rows.append({
        'strategy_key':   STRATEGY_KEY,
        'focal_gamma':    FOCAL_GAMMA,
        'focal_beta':     FOCAL_BETA,
        'label_smoothing': FOCAL_LABEL_SMOOTHING,
        'run':            r['run'],
        'seed':           r['seed'],
        'accuracy':       r['accuracy'],
        'weighted_f1':    r['f1_score'],
        'macro_f1':       r['macro_f1'],
        'balanced_acc':   r['balanced_acc'],
        'recall_minority': r['recall_minority'],
    })

pd.DataFrame(summary_rows).to_csv(
    os.path.join(BASE_RESULT_DIR, 'strategy_summary_v2.csv'), index=False)
print(f'\nSaved → strategy_summary_v2.csv')

In [ ]:
# ========== 11. LATEX TABLE FOR PAPER ========== #

minority_short = minority_name.replace('_Garlic', '').replace('_', ' ')

latex = r"""\begin{table}[h]
\centering
\caption{Ablation Study: Effect of Fixing Label Smoothing Conflict in CB Focal Loss.
All models use EfficientNetB4 + FSDA (blocks 3-7 unfrozen). Results are
Mean $\pm$ Std over 3 independent seeds. $\star$ denotes primary metric.}
\label{tab:ablation_cbfocal}
\begin{tabular}{lcccc}
\hline
\textbf{Configuration} & \textbf{Weighted F1} & \textbf{Macro F1$\star$} & \textbf{Balanced Acc} & \textbf{Recall (Partial.)} \\
\hline
"""

configs = [
    ('FSDA + CE (Baseline)', BASELINE_WEIGHTED_F1, BASELINE_MACRO_F1, BASELINE_BALANCED_ACC, BASELINE_RECALL_MINORITY,
     0, 0, 0, 0),
    ('FSDA + CB Focal v1\\textsuperscript{a}\n(broken: LS=0.1, $\\gamma$=2.0)',
     V1_WEIGHTED_F1, V1_MACRO_F1, V1_BALANCED_ACC, V1_RECALL_MINORITY, 0, 0, 0, 0),
    ('\\textbf{FSDA + CB Focal v2}\n(fixed: LS=0.0, $\\gamma$=3.0)',
     v2_weighted_f1, v2_macro_f1, v2_balanced_acc, v2_recall_minority,
     v2_weighted_f1_std, v2_macro_f1_std, v2_balanced_acc_std, v2_recall_minority_std),
]

for (name, wf1, mf1, ba, rm, wf1s, mf1s, bas, rms) in configs:
    def fmt(v, s):
        return f'{v:.4f} $\\pm$ {s:.4f}' if s > 0 else f'{v:.4f}'
    latex += f'{name} & {fmt(wf1,wf1s)} & {fmt(mf1,mf1s)} & {fmt(ba,bas)} & {fmt(rm,rms)} \\\\\n'

latex += r"""\hline
\end{tabular}
\begin{tablenotes}
  \small
  \item[a] v1 uses label\_smoothing=0.1 inside the focal loss, creating gradient
           conflict: smoothing artificially raises $p_t$, reducing $(1-p_t)^\gamma$
           and weakening focus on minority class samples.
  \item $\star$ Macro F1 is the primary metric for imbalanced datasets (IR$\approx$3.43).
\end{tablenotes}
\end{table}
"""

print(latex)

with open(os.path.join(BASE_RESULT_DIR, 'latex_ablation_table.tex'), 'w') as f:
    f.write(latex)
print('Saved → latex_ablation_table.tex')

In [ ]:
# ========== 12. AGGREGATE CONFUSION MATRIX ========== #

agg_cm = np.zeros((len(class_names), len(class_names)), dtype=int)
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(agg_cm, annot=True, fmt='d',
            xticklabels=class_names, yticklabels=class_names,
            cmap='Blues', ax=axes[0])
axes[0].set_title('Aggregate Confusion Matrix (raw, 3 runs)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

cm_norm = agg_cm.astype(float) / agg_cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=class_names, yticklabels=class_names,
            cmap='Blues', ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Normalised Confusion Matrix (row = recall per class)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle(f'{STRATEGY_LABEL}\nAggregate over {N_RUNS} runs',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'agg_confusion_matrix.png'), dpi=300)
plt.show()

print('Per-class Recall (diagonal of normalised CM):')
for i, cn in enumerate(class_names):
    marker = ' ← minority' if 'Partial' in cn else ''
    print(f'  {cn:<30}: {cm_norm[i,i]:.4f}{marker}')

In [ ]:
# ========== 13. ROC CURVES ========== #

all_y_true = np.concatenate([r['y_true']    for r in all_runs_results])
all_probs  = np.concatenate([r['pred_probs'] for r in all_runs_results])
nc         = len(class_names)
y_bin      = label_binarize(all_y_true, classes=range(nc))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tab_cols  = plt.cm.tab10.colors
auc_scores = {}
fpr_d, tpr_d = {}, {}

for i, cn in enumerate(class_names):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cn] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_cols[i], lw=2,
                 label=f'{cn}  (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0,1], ylim=[0,1.01],
            xlabel='FPR', ylabel='TPR', title='Per-Class ROC (OvR, aggregate)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(nc)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(nc):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= nc
macro_auc = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg AUC={macro_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0,1], ylim=[0,1.01],
            xlabel='FPR', ylabel='TPR', title='Macro-Average ROC')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'ROC Curves — {STRATEGY_LABEL}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300)
plt.show()

for cn, av in auc_scores.items():
    print(f'  AUC {cn:<30}: {av:.4f}')
print(f'  Macro-average AUC        : {macro_auc:.4f}')

In [ ]:
# ========== 14. ZIP ALL RESULTS ========== #
import shutil

zip_out = f'/kaggle/working/{STRATEGY_KEY}'
shutil.make_archive(zip_out, 'zip', BASE_RESULT_DIR)
zip_sz = os.path.getsize(f'{zip_out}.zip') / (1024*1024)

print(f'\nDone! Archive: {zip_out}.zip  ({zip_sz:.2f} MB)')
print(f'\nKey files generated:')
for fname in sorted(os.listdir(BASE_RESULT_DIR)):
    fpath = os.path.join(BASE_RESULT_DIR, fname)
    if os.path.isfile(fpath):
        sz = os.path.getsize(fpath)
        print(f'  {fname:<50} ({sz/1024:.1f} KB)')
    else:
        print(f'  {fname}/ (folder)')